In [ ]:
# ------------------------------------------------------------
# IMPORTS GENERALES
# ------------------------------------------------------------
import pandas as pd
import numpy as np
from sklearn.metrics import (
    confusion_matrix, accuracy_score, precision_score,
    recall_score, f1_score, classification_report
)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.model_selection import train_test_split
from sklearn.inspection import permutation_importance
from sklearn.metrics import roc_auc_score
import matplotlib.pyplot as plt
import joblib

# ------------------------------------------------------------
# CARGA DE DATOS Y MODELO INICIAL
# (Rutas generales: reemplazar cuando tengas los archivos)
# ------------------------------------------------------------
df_train = pd.read_csv("data_train.csv")     # Reemplazar
df_val = pd.read_csv("data_val.csv")         # Reemplazar

X_train = df_train.drop("target", axis=1)
y_train = df_train["target"]

X_val = df_val.drop("target", axis=1)
y_val = df_val["target"]

model_v1 = joblib.load("model_v1.joblib")    # Modelo inicial

# ------------------------------------------------------------
# 1. EVALUACIÓN CUANTITATIVA – Métricas básicas
# ------------------------------------------------------------
y_pred = model_v1.predict(X_val)

cm = confusion_matrix(y_val, y_pred)
acc = accuracy_score(y_val, y_pred)
prec = precision_score(y_val, y_pred)
rec = recall_score(y_val, y_pred)
f1 = f1_score(y_val, y_pred)

print("Matriz de Confusión:\n", cm)
print("\nAccuracy:", acc)
print("Precision:", prec)
print("Recall:", rec)
print("F1-score:", f1)
print("\nReporte completo:\n", classification_report(y_val, y_pred))

# Aquí puedes escribir tu análisis interpretando:
# - Por qué accuracy no es adecuada en reingresos
# - Por qué F1 o Recall son mejores métricas en este contexto

# ------------------------------------------------------------
# 2. INTERPRETABILIDAD – Árbol simple o importancia
# ------------------------------------------------------------

# Opción A: Mostrar el árbol del modelo si es un Decision Tree
plt.figure(figsize=(12, 8))
plot_tree(model_v1, feature_names=X_train.columns, filled=True)
plt.show()

# Opción B: Importancia de características
importancias = model_v1.feature_importances_
feat_imp = pd.DataFrame({"feature": X_train.columns, "importance": importancias})
feat_imp = feat_imp.sort_values("importance", ascending=False)
print(feat_imp)

# Opción C: Permutation Importance (más general)
perm = permutation_importance(model_v1, X_val, y_val, n_repeats=5)
print("Permutation Importance:")
print(perm)

# Añade explicación del funcionamiento global del modelo

# ------------------------------------------------------------
# 3. JUSTICIA – Fairness con respecto a 'gender'
# ------------------------------------------------------------
# Suponiendo que existe una columna 'gender'
sensitive_attr = df_val["gender"]

# Ejemplo: Diferencia en tasas de predicción positiva (DPD)
positive_rate_group0 = y_pred[sensitive_attr == 0].mean()
positive_rate_group1 = y_pred[sensitive_attr == 1].mean()

dpd = abs(positive_rate_group0 - positive_rate_group1)

print("Positive Prediction Rate Group 0:", positive_rate_group0)
print("Positive Prediction Rate Group 1:", positive_rate_group1)
print("DPD (Demographic Parity Difference):", dpd)

# Puedes comparar con threshold aceptables (como ±0.1)

# ------------------------------------------------------------
# 4. A/B TESTING – Segunda versión del modelo
# ------------------------------------------------------------
# Crear un modelo nuevo (ejemplo: ajustar hiperparámetros)
model_v2 = DecisionTreeClassifier(
    max_depth=5,
    min_samples_leaf=10,
    class_weight="balanced"
)

model_v2.fit(X_train, y_train)

y_pred_v2 = model_v2.predict(X_val)

# Evaluar con la métrica recomendada (F1 probablemente)
f1_v1 = f1_score(y_val, y_pred)
f1_v2 = f1_score(y_val, y_pred_v2)

print("F1 modelo v1:", f1_v1)
print("F1 modelo v2:", f1_v2)

# A/B testing: diferencia en rendimiento
delta = f1_v2 - f1_v1
print("Mejora (ΔF1):", delta)

# ------------------------------------------------------------
# 5. DESPLIEGUE – Esqueleto en código (simbolico)
# ------------------------------------------------------------
print("""
Estrategia recomendada:
- Canary Release o Shadow Deployment
(Explicar según el contexto del hospital)
""")


Una estrategia de despliegue gradual, como canary release o shadow deployment, es la más adecuada para la Clínica de los Alpes porque permite validar el comportamiento del modelo con datos reales del hospital sin poner en riesgo a los pacientes ni alterar abruptamente los flujos de trabajo del personal médico. En un entorno clínico, donde las decisiones automatizadas pueden influir directamente en la calidad de la atención, es indispensable introducir el nuevo modelo de manera controlada: primero operando en paralelo con el modelo actual para comparar predicciones en tiempo real sin impactar decisiones, y luego exponiéndolo progresivamente a un porcentaje limitado de casos. Esto reduce la probabilidad de errores inesperados, permite monitorear métricas críticas (como recall, que es clave para no dejar pasar pacientes de alto riesgo), y ofrece evidencia sólida antes de reemplazar completamente la versión anterior, respetando así las restricciones operativas y el principio de “no hacer daño” en sistemas médicos.

In [ ]:
# Calcular la métrica en tu conjunto de validación Usas las predicciones del modelo inicial y la columna gender.

y_pred = model_v1.predict(X_val)
gender = df_val["gender"]

positive_rate_0 = y_pred[gender == 0].mean()
positive_rate_1 = y_pred[gender == 1].mean()

dpd = abs(positive_rate_0 - positive_rate_1)

print("Positive rate grupo 0:", positive_rate_0)
print("Positive rate grupo 1:", positive_rate_1)
print("Demographic Parity Difference:", dpd)



3. Interpretar el resultado

Debes escribir algo como:

Si DPD es pequeño (< 0.1):
“El modelo no muestra un sesgo significativo entre géneros, por lo que se considera justo bajo la métrica de Demographic Parity.”

Si es grande:
“El modelo predice más positivos para un género que para otro, lo cual es problemático porque implicaría que la herramienta está clasificando de manera desigual a pacientes con características clínicas similares.”

4. Justificar por qué esta métrica es apropiada en este caso

Puedes escribir:

“El propósito del modelo es identificar pacientes con alto riesgo de reingreso. Un modelo injusto podría sobre-diagnosticar o sub-diagnosticar a un género, lo cual sería éticamente inaceptable en un entorno clínico. Por eso es importante medir la paridad en las tasas de predicción positiva.”

In [ ]:
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import f1_score, recall_score, precision_score, accuracy_score

# ---------------------------
# Versión 1: modelo original
# ---------------------------
model_v1 = clf              # tu árbol original
y_pred_v1 = model_v1.predict(X_val)

f1_v1      = f1_score(y_val, y_pred_v1)
recall_v1  = recall_score(y_val, y_pred_v1)
prec_v1    = precision_score(y_val, y_pred_v1)
acc_v1     = accuracy_score(y_val, y_pred_v1)

print("V1 - Accuracy:", acc_v1)
print("V1 - Precision:", prec_v1)
print("V1 - Recall:", recall_v1)
print("V1 - F1:", f1_v1)

# ---------------------------
# Versión 2: árbol ajustado
#   - más podado
#   - maneja desbalance
# ---------------------------
model_v2 = DecisionTreeClassifier(
    criterion="gini",       # podrías probar "entropy"
    max_depth=10,           # menor que 20 para reducir overfitting
    min_samples_leaf=10,    # más hojas mínimas = árbol más simple
    class_weight="balanced",
    random_state=2
)

model_v2.fit(X_train, y_train)

y_pred_v2 = model_v2.predict(X_val)

f1_v2      = f1_score(y_val, y_pred_v2)
recall_v2  = recall_score(y_val, y_pred_v2)
prec_v2    = precision_score(y_val, y_pred_v2)
acc_v2     = accuracy_score(y_val, y_pred_v2)

print("\nV2 - Accuracy:", acc_v2)
print("V2 - Precision:", prec_v2)
print("V2 - Recall:", recall_v2)
print("V2 - F1:", f1_v2)

# ---------------------------
# Comparación tipo A/B
# (usa la métrica principal que definiste en el punto 1)
# ---------------------------
print("\nComparación por F1:")
print("F1 V1:", f1_v1)
print("F1 V2:", f1_v2)
print("ΔF1 (V2 - V1):", f1_v2 - f1_v1)
